In [1]:
import datasets
import json

eval_set = datasets.load_dataset("Open-Style/Open-LLM-Benchmark", "questions")
grouped_datasets = {}
for i, example in enumerate(eval_set['train']):
    dataset = example["dataset"]
    # Add a unique ID if one doesn't exist
    if "id" not in example:
        example = dict(example)  # Create a mutable copy
        example["id"] = f"{dataset}_{i}"  # Add a synthetic ID
        
    if dataset not in grouped_datasets:
        grouped_datasets[dataset] = []
    grouped_datasets[dataset].append(example)

In [2]:
grouped_datasets.keys()

dict_keys(['ARC', 'CommonsenseQA', 'Hellaswag', 'MMLU', 'MedMCQA', 'OpenbookQA', 'Winogrande', 'piqa', 'race'])

In [3]:
# Check the structure of one example to see available fields
print("Sample keys:", list(eval_set['train'][0].keys()))

Sample keys: ['question', 'answerKey', 'options', 'first_filter', 'passage', 'second_filter', 'dataset']


In [4]:
from transformers import AutoTokenizer,AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2", device_map="auto")

tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2")

# Ensure model is running on the GPU if available
device = model.device

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [18]:
def create_few_shot_prompt(sample, dataset_name):
    """
    Create a few-shot prompt with examples from the same dataset
    """
    # Get 3 examples from the dataset (different from current sample)
    examples = [ex for ex in grouped_datasets[dataset_name] 
                if ex['id'] != sample['id']][:5]
                
    # Build the few-shot prompt
    prompt = "Please answer multiple-choice questions by selecting the best option from the options provided.\n\n"
    
    # Add few-shot examples
    for i, example in enumerate(examples):
        prompt += f"Question {i+1}: {example['question']}\n"
        for option in example['options']:
            prompt += f"{option['label']}: {option['text']}\n"
        prompt += f"The answer is {example['answerKey']}.\n\n"
    
    # Add the current question
    prompt += f"Question: {sample['question']}\n"
    for option in sample['options']:
        prompt += f"{option['label']}: {option['text']}\n"
    prompt += "The answer is "
    
    return prompt

In [5]:
def create_few_shot_prompt2(sample, dataset_name):
    """
    Create a few-shot prompt with examples from the same dataset
    """
    # Task-specific instructions based on dataset
    task_instructions = {
        "CommonsenseQA": "Please answer multiple-choice questions using common sense reasoning. For each question, consider what a person would naturally know about everyday situations and objects.",
        
        "OpenBookQA": "Please answer multiple-choice science questions using basic science facts. For each question, please use elementary science knowledge and apply it to the specific situation.",
        
        "PIQA": "Please answer questions about physical common sense. For each question, choose the option that is more physically plausible based on how the physical world works.",
        
        # Add more datasets and their specific instructions as needed
        "default": "I'll answer multiple-choice questions by selecting the best option."
    }
    
    # Get the appropriate instruction for this dataset
    instruction = task_instructions.get(dataset_name, task_instructions["default"])
    
    # Build the prompt starting with the instruction
    prompt = f"{instruction}\n\n"
    
    # Get 3 examples from the dataset (different from current sample)
    examples = [ex for ex in grouped_datasets[dataset_name] 
                if ex['id'] != sample['id']][:3]
                
    # Add few-shot examples
    for i, example in enumerate(examples):
        prompt += f"Question {i+1}: {example['question']}\n"
        for option in example['options']:
            prompt += f"{option['label']}: {option['text']}\n"
    #    prompt += f"The answer is {example['answerKey']}.\n\n"
     
            # Add reasoning for the examples (this is optional but helpful)
            if dataset_name == "CommonsenseQA":
                prompt += "Reasoning: Using my common sense knowledge of everyday situations, "
            elif dataset_name == "OpenbookQA":
                prompt += "Reasoning: Based on elementary science facts, "
            elif dataset_name == "piqa":
                prompt += "Reasoning: Considering physical plausibility, "
            else:
                prompt += "Reasoning: "
            # Add a simple reasoning template
            correct_option = next(opt for opt in example['options'] if opt['label'] == example['answerKey'])
            prompt += f"option {example['answerKey']} ({correct_option['text']}) is correct because it aligns with everyday knowledge.\n"
            prompt += f"The answer is {example['answerKey']}.\n\n"
    
    # Add the current question
    prompt += f"Question: {sample['question']}\n"
    for option in sample['options']:
        prompt += f"{option['label']}: {option['text']}\n"
    
    # Add a prompt for reasoning before the answer
    if dataset_name == "CommonsenseQA":
        prompt += "Reasoning: Using my common sense knowledge, "
    elif dataset_name == "OpenbookQA":
        prompt += "Reasoning: Based on elementary science facts, "
    elif dataset_name == "piqa":
        prompt += "Reasoning: Considering physical plausibility, "
    else:
        prompt += "Reasoning: "
    
    return prompt

In [6]:
import re
def extract_answer(output, sample):
    """
    Extract answer from model output with robust pattern matching
    """
    # Try different patterns to extract answer
    answer_patterns = [
        r"The answer is\s*([A-E])[.)]?",  # "The answer is A." or "The answer is A)"
        r"([A-E])[.)]?\s*is correct",      # "A. is correct" or "A) is correct"
        r"answer[: ]+([A-E])[.)]?",        # "answer: A" or "answer: A."
        r"^([A-E])[.)]?$",                 # Just "A" or "A."
    ]
    
    for pattern in answer_patterns:
        matches = re.search(pattern, output, re.IGNORECASE)
        if matches:
            return matches.group(1)
    
 # If no pattern matches, check if any option label is in the output
    valid_options = [opt['label'] for opt in sample['options']]
    for option in valid_options:
        if f" {option} " in f" {output} " or f" {option}." in f" {output} " or f" {option}," in f" {output} ":
            return option
            
    # Return the first character as last resort
    if output and output[0].upper() in valid_options:
        return output[0].upper()

In [7]:
# Define a function for inference
def infer_llm(sample,dataset_name):
    #question = sample["question"]
    #options = sample["options"]
    #prompt = f"{question}\n"
    #for option in options:
    #    prompt += f"{option['label']}: {option['text']}\n"

    ### You can change the prompt to to suit the model you are using.
    # Example:
    # Answer in A/B/C/D:
    # Answer in a single word or phrase:

    #prompt += "Answer: "
    prompt = create_few_shot_prompt(sample, dataset_name)
    
    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate the output
    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=50,  
        num_return_sequences=1,
        attention_mask=inputs["attention_mask"],
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.1,
        do_sample=True,
    )

    # Decode the output
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    #answer = decoded_output.split("Answer:")[-1].strip()

    # Extract the answer part
    answer_part = decoded_output.replace(prompt, "")  # Get just the generated part
    
    # Extract the actual answer letter
    answer = extract_answer(answer_part, sample)
    
    return answer, decoded_output

# Define the evaluation function
def evaluate_samples(dataset_name):
    samples = grouped_datasets[dataset_name]
    correct = 0
    total = len(samples)
    predictions = []

    print(f"Evaluating {dataset_name} with {total} samples")
    
    for i, sample in enumerate(samples):
        if i % 50 == 0:
            print(f"Processing {i}/{total}...")
        
        predicted_answer, full_output = infer_llm(sample, dataset_name)
        predictions.append(predicted_answer)

        if predicted_answer == sample["answerKey"]:
            correct += 1

    accuracy = correct / total * 100
    return predictions, accuracy


In [ ]:
# Evaluate the samples simple few_shot_prompt
predictions, accuracy = evaluate_samples('CommonsenseQA')
print(f"Accuracy for CommonsenseQA: {accuracy:.2f}%")

In [ ]:
# Evaluate the samples simple few_shot_prompt
predictions, accuracy = evaluate_samples('OpenbookQA')
print(f"Accuracy for OpenbookQA: {accuracy:.2f}%")

In [ ]:
# Evaluate the samples simple few_shot_prompt
predictions, accuracy = evaluate_samples('piqa')
print(f"Accuracy for PiQA: {accuracy:.2f}%")

In [ ]:
Update to call few_shot_prompt2

In [ ]:
# Evaluate the samples simple few_shot_prompt2
predictions, accuracy = evaluate_samples('CommonsenseQA')
print(f"Accuracy for CommonsenseQA: {accuracy:.2f}%")

In [ ]:
# Evaluate the samples simple few_shot_prompt2
predictions, accuracy = evaluate_samples('OpenbookQA')
print(f"Accuracy for OpenbookQA: {accuracy:.2f}%")

In [ ]:
# Evaluate the samples simple few_shot_prompt2
predictions, accuracy = evaluate_samples('piqa')
print(f"Accuracy for PiQA: {accuracy:.2f}%")

In [ ]:
Previous runs

In [8]:
# Evaluate the samples
predictions, accuracy = evaluate_samples('CommonsenseQA')
print(f"Accuracy for CommonsenseQA: {accuracy:.2f}%")

Evaluating CommonsenseQA with 710 samples
Processing 0/710...
Processing 10/710...
Processing 20/710...
Processing 30/710...
Processing 40/710...
Processing 50/710...
Processing 60/710...
Processing 70/710...
Processing 80/710...
Processing 90/710...
Processing 100/710...
Processing 110/710...
Processing 120/710...
Processing 130/710...
Processing 140/710...
Processing 150/710...
Processing 160/710...
Processing 170/710...
Processing 180/710...
Processing 190/710...
Processing 200/710...
Processing 210/710...
Processing 220/710...
Processing 230/710...
Processing 240/710...
Processing 250/710...
Processing 260/710...
Processing 270/710...
Processing 280/710...
Processing 290/710...
Processing 300/710...
Processing 310/710...
Processing 320/710...
Processing 330/710...
Processing 340/710...
Processing 350/710...
Processing 360/710...
Processing 370/710...
Processing 380/710...
Processing 390/710...
Processing 400/710...
Processing 410/710...
Processing 420/710...
Processing 430/710...
P

In [10]:

predictions, accuracy = evaluate_samples('OpenbookQA')
print(f"Accuracy for OpenbookQA: {accuracy:.2f}%")
#print(f"Accuracy for PiQA: {accuracy_piqa:.2f}%")

Evaluating OpenbookQA with 491 samples
Processing 0/491...
Processing 10/491...
Processing 20/491...
Processing 30/491...
Processing 40/491...
Processing 50/491...
Processing 60/491...
Processing 70/491...
Processing 80/491...
Processing 90/491...
Processing 100/491...
Processing 110/491...
Processing 120/491...
Processing 130/491...
Processing 140/491...
Processing 150/491...
Processing 160/491...
Processing 170/491...
Processing 180/491...
Processing 190/491...
Processing 200/491...
Processing 210/491...
Processing 220/491...
Processing 230/491...
Processing 240/491...
Processing 250/491...
Processing 260/491...
Processing 270/491...
Processing 280/491...
Processing 290/491...
Processing 300/491...
Processing 310/491...
Processing 320/491...
Processing 330/491...
Processing 340/491...
Processing 350/491...
Processing 360/491...
Processing 370/491...
Processing 380/491...
Processing 390/491...
Processing 400/491...
Processing 410/491...
Processing 420/491...
Processing 430/491...
Proc

In [11]:
predictions, accuracy = evaluate_samples('piqa')
#print(f"Accuracy for OpenbookQA: {accuracy_oqa:.2f}%")
print(f"Accuracy for PiQA: {accuracy:.2f}%")

Evaluating piqa with 696 samples
Processing 0/696...
Processing 10/696...
Processing 20/696...
Processing 30/696...
Processing 40/696...
Processing 50/696...
Processing 60/696...
Processing 70/696...
Processing 80/696...
Processing 90/696...
Processing 100/696...
Processing 110/696...
Processing 120/696...
Processing 130/696...
Processing 140/696...
Processing 150/696...
Processing 160/696...
Processing 170/696...
Processing 180/696...
Processing 190/696...
Processing 200/696...
Processing 210/696...
Processing 220/696...
Processing 230/696...
Processing 240/696...
Processing 250/696...
Processing 260/696...
Processing 270/696...
Processing 280/696...
Processing 290/696...
Processing 300/696...
Processing 310/696...
Processing 320/696...
Processing 330/696...
Processing 340/696...
Processing 350/696...
Processing 360/696...
Processing 370/696...
Processing 380/696...
Processing 390/696...
Processing 400/696...
Processing 410/696...
Processing 420/696...
Processing 430/696...
Processing

In [ ]:
# Example usage
for dataset in ["CommonsenseQA", "openBookQA", "piqa"]:
    if dataset in grouped_datasets:
        predictions, accuracy = evaluate_samples(dataset)
        print(f"Accuracy for {dataset}: {accuracy:.2f}%")
        
        # Save results for analysis
        with open(f"{dataset}_results.json", "w") as f:
            results = {
                "accuracy": accuracy,
                "predictions": predictions
            }
            json.dump(results, f)